In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualização gráfica inline no Jupyter notebook
%matplotlib inline

# Extrair características de ensaios reais de EEG

Calcula potências de bandas e preserva os rótulos e identidades necessários para o tutorial 42.

Essas gravações reais de SSVEP de Nakanishi2015 são distribuídas como a versão processada
[nm000118](https://nemar.org/dataset/nm000118)
([estudo](https://doi.org/10.1371/journal.pone.0140703)).
Filtragem, redução de taxa de amostragem (*downsampling*) e tratamento de latência já foram aplicados;
não desloque os inícios dos eventos novamente. CPU é suficiente. Conexão à Internet é necessária
para o primeiro download; ``EEGDASH_CACHE_DIR`` mantém os downloads entre as execuções.

O subconjunto explícito usa 3 participantes, cerca de 21.1 MB de arquivos de sinal.

## Antes de começar
Instale o EEGDash e suas dependências. O tutorial 02 explica janelas, e o tutorial 11
explica por que os identificadores de participante são importantes. Esta página é executada
de forma independente e depois salva ``plot_40_features.csv`` e um esquema JSON para o tutorial 42.
Mantenha ambos os arquivos no mesmo diretório de cache.


In [ ]:
# Importa módulos de sistema e caminhos de arquivo
import os
from pathlib import Path

# Importa bibliotecas para plotagem, computação matricial e análise de dados tabulares
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa gerador de janelas baseadas em eventos da Braindecode
from braindecode.preprocessing import create_windows_from_events

# Importa classes de dataset e extratores de características do EEGDash
from eegdash import EEGDashDataset
from functools import partial
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_bands_power,
    spectral_preprocessor,
)
# Importa biblioteca para serialização em JSON
import json

## 1. Carregar e inspecionar as gravações selecionadas
A tarefa é a classificação de frequência SSVEP, mas aqui o objetivo imediato é uma
tabela de características legível. As três gravações compartilham oito canais posteriores
e uma taxa de 256 Hz. A ordenação numérica das strings de anotação fornece um mapeamento
de classes estável; esses rótulos descrevem o estímulo focado, não o estado dos olhos.



In [ ]:
# Define o diretório de cache a partir da variável de ambiente ou usa o padrão local
cache_dir = Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache"))
# Define os três sujeitos a serem carregados para a demonstração
subjects = ["1", "2", "3"]
# Carrega o conjunto de dados SSVEP da primeira sessão e primeira execução (run 0)
dataset = EEGDashDataset(
    cache_dir=cache_dir,
    dataset="nm000118",
    subject=subjects,
    session="0",
    run="0",
    task="ssvep",
    n_jobs=1,
)
# Valida que os 3 sujeitos foram carregados no dataset
assert len(dataset.datasets) == len(subjects)
# Exibe os metadados de sujeito, sessão e execução do dataset
print(dataset.description[["subject", "session", "run"]])
# Acessa a primeira gravação para extrair a taxa de amostragem e os canais
raw = dataset.datasets[0].raw
sfreq = raw.info["sfreq"]
channel_names = raw.ch_names
# Identifica as classes de frequência a partir das anotações ordenadas numericamente
class_names = sorted(set(raw.annotations.description), key=float)
# Cria o mapeamento de nome da classe para índice numérico de 0 a 11
mapping = {name: index for index, name in enumerate(class_names)}
# Confirma a existência de exatamente 12 classes de estímulo SSVEP
assert len(mapping) == 12
# Valida consistência de canais, taxa de amostragem e anotações em todas as gravações
for recording in dataset.datasets:
    assert recording.raw.ch_names == channel_names
    assert recording.raw.info["sfreq"] == sfreq
    assert set(recording.raw.annotations.description) == set(mapping)
# Imprime informações de canais, taxa e frequências de estímulo encontradas
print(f"Channels: {channel_names}; sampling rate: {sfreq} Hz")
print("Observed stimulus frequencies (Hz):", class_names)

## 2. Janelar os ensaios observados
Quatro segundos fornecem amostras suficientes para segmentos espectrais repetidos de um segundo.
Tamanho e passo (*stride*) iguais, juntamente com o descarte do restante final, retêm
uma janela por ensaio anotado. Assim, as linhas da tabela de características podem ser rastreadas
de volta a uma gravação e amostra inicial em vez de se tornarem vetores anônimos.
O evento original dura 4.15 segundos; seus 0.15 segundos finais não são usados.



In [ ]:
# Define o tamanho da janela em amostras correspondente a 4 segundos (4 * 256 = 1024 amostras)
window_size = int(4 * sfreq)
# Cria as janelas a partir das anotações de estímulo descartando sobras com 'on_last_window="drop"'
windows = create_windows_from_events(
    dataset,
    mapping=mapping,
    window_size_samples=window_size,
    window_stride_samples=window_size,
    on_last_window="drop",
    preload=True,
)
# Extrai os metadados das janelas e redefine o índice para alinhamento limpo
metadata = windows.get_metadata().reset_index(drop=True)
# Obtém o vetor de rótulos numéricos como array numpy de inteiros
y = metadata["target"].to_numpy(dtype=int)
# Assegura correspondência exata entre número de janelas e metadados
assert len(windows) == len(metadata)
# Assegura que todas as 12 classes estão representadas no vetor alvo y
assert set(y) == set(mapping.values())
# Assegura que não há janelas duplicadas para o mesmo sujeito e mesmo instante de início
assert not metadata.duplicated(["subject", "session", "run", "i_start_in_trial"]).any()
# Exibe tabela de contingência cruzando sujeito por classe de estímulo
print(pd.crosstab(metadata["subject"], y))

## 3. Calcular potências de bandas nomeadas com um espectro de Welch compartilhado
``FeatureExtractor`` executa primeiro ``spectral_preprocessor`` e passa sua grade de frequências
e valores de potência para a função de potência em bandas. ``partial`` fixa a taxa de amostragem
escolhida e a faixa de frequência sem a necessidade de definir uma função wrapper.
``nperseg=int(sfreq)`` usa segmentos de Welch de um segundo, resultando em uma grade de 1 Hz.
A função soma os valores espectrais dentro de cada banda nomeada. Mantenha essa grade
fixa ao comparar magnitudes entre tabelas.

Três bandas vezes oito canais produzem 24 colunas de características por ensaio.
``batch_size=64`` controla a memória de extração, não a duração do ensaio ou o treinamento
do modelo. Bandas largas resumem a potência, mas descartam os detalhes finos de frequência
que distinguem classes de SSVEP próximas; portanto, um decodificador final mais fraco é
possível mesmo quando a extração está correta.

As colunas de características numéricas são capturadas antes da adição dos metadados. Sujeito,
sessão, execução (*run*), amostra inicial e alvo devem acompanhar cada linha para avaliação agrupada
posterior; ``frequency_hz`` torna o significado do alvo legível por humanos.
Ela nunca deve ser incluída como um preditor desse mesmo alvo.



In [ ]:
# Dicionário com os intervalos das bandas de frequência de interesse: teta, alfa e beta
bands = {"theta": (4, 8), "alpha": (8, 12), "beta": (12, 30)}
# Configura o extrator espectral associando a função de potência em bandas ao pré-processador Welch
spectral = FeatureExtractor(
    {"power": partial(spectral_bands_power, bands=bands)},
    preprocessor=partial(
        spectral_preprocessor, fs=sfreq, nperseg=int(sfreq), f_min=4, f_max=30
    ),
)
# Executa a extração em lote para todas as janelas criadas
feature_dataset = extract_features(
    windows, {"spectral": spectral}, batch_size=64, n_jobs=1
)
# Converte o conjunto de características resultante em um DataFrame do pandas
feature_table = feature_dataset.to_dataframe()
# Armazena os nomes das colunas de características numéricas
feature_columns = list(feature_table.columns)
# Valida que o número de características corresponde a 3 bandas * 8 canais = 24 colunas
assert len(feature_columns) == len(bands) * len(channel_names)
# Garante que todos os valores calculados são finitos (sem NaN ou infinitos)
assert np.isfinite(feature_table.to_numpy()).all()
# Os metadados das janelas carregam a identidade do participante e do ensaio. Una explicitamente:
# o DataFrame padrão da extração de características contém apenas os valores das características.
feature_table = pd.concat(
    [
        metadata[["subject", "session", "run", "i_start_in_trial", "target"]],
        feature_table,
    ],
    axis=1,
)
# Adiciona o valor real da frequência em Hz correspondente à classe para inspeção humana
feature_table["frequency_hz"] = [float(class_names[target]) for target in y]
# Valida ausência de duplicidade de identificação do ensaio na tabela final
assert not feature_table.duplicated(
    ["subject", "session", "run", "i_start_in_trial"]
).any()

## 4. Persistir a tabela exata e o contrato de colunas para o tutorial 42
O formato CSV torna a transferência inspecionável sem um mecanismo de armazenamento adicional.
O arquivo JSON fornece a ordem autoritativa das colunas de características, mapeamento e parâmetros
de extração. Leia os identificadores BIDS explicitamente como strings: caso contrário,
um leitor de CSV pode reinterpretar ``"0"`` ou um identificador com zeros à esquerda como um número.
As asserções de recarregamento verificam os valores e rótulos reais salvos, em vez de assumir que
uma gravação bem-sucedida preservou o contrato da tabela.



In [ ]:
# Define o caminho de saída para o arquivo CSV de características
output_path = cache_dir / "plot_40_features.csv"
# Salva a tabela completa em CSV sem incluir o índice numérico padrão
feature_table.to_csv(output_path, index=False)
# Salva o arquivo de esquema JSON associado com metadados cruciais para o próximo tutorial
(output_path.with_suffix(".json")).write_text(
    json.dumps(
        {
            "dataset": "nm000118",
            "subjects": subjects,
            "session": "0",
            "run": "0",
            "task": "ssvep",
            "mapping": mapping,
            "feature_columns": feature_columns,
            "sfreq": sfreq,
            "window_samples": window_size,
            "bands": bands,
        },
        indent=2,
    )
)
# Recarrega os dados do CSV forçando os identificadores como strings para validar a persistência
roundtrip = pd.read_csv(output_path, dtype={"subject": str, "session": str, "run": str})
# Assegura que os valores numéricos salvos e recarregados são idênticos
np.testing.assert_allclose(roundtrip[feature_columns], feature_table[feature_columns])
# Assegura integridade exata dos alvos salvos
np.testing.assert_array_equal(roundtrip["target"], y)
# Imprime mensagem de confirmação com caminho e dimensões da tabela gravada
print("Saved:", output_path.resolve(), feature_table.shape)

## 5. Inspecionar as distribuições das características medidas
As características salvas permanecem como valores lineares de potência. Apenas este gráfico aplica log10,
de modo que uma diferença vertical de uma unidade representa um fator de dez na potência.
Gráficos boxplot resumem a distribuição dos ensaios de cada coluna banda/canal;
``showfliers=False`` oculta marcadores fora dos limites dos bigodes, mas não remove esses ensaios da tabela.
Colunas muito baixas ou constantes merecem inspeção do sinal antes da interpretação.

Execute o tutorial 42 com o mesmo cache para ajustar modelos diretamente a partir desses arquivos.
Se adicionar uma família de características, gere novamente o CSV e o JSON para que a próxima
página use os novos nomes de características e nunca treine acidentalmente em metadados.



In [ ]:
# Aplica transformação log10 com piso de segurança para fins visuais
log_power = np.log10(np.maximum(feature_table[feature_columns], 1e-30))
# Cria figura para plotagem dos boxplots das características
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
# Plota boxplot para cada coluna banda/canal ocultando outliers pontuais
ax.boxplot(log_power.to_numpy(), tick_labels=feature_columns, showfliers=False)
# Rotaciona os rótulos do eixo X em 90 graus para facilitar leitura
ax.tick_params(axis="x", rotation=90)
# Define rótulo do eixo Y e título do gráfico
ax.set(ylabel="log10 band power", title="Recorded SSVEP trials: channel-wise features")
# Exibe o gráfico
plt.show()